# Baseline model

Predicting whether an unplanned New York hospital admission lasts more than
7 days. Decisions are in `docs/notes/2026-11-05-data-cohort-split.md`.

In [1]:
import re
import urllib.request
from pathlib import Path

import pandas as pd

SEED = 42

## Step 1: Get the data

Download the 2023 and 2024 SPARCS files unless they're already there, then
stop with an error if either file isn't the expected size.

In [2]:
RAW_DIR = Path("data/raw")
URL = "https://health.data.ny.gov/api/views/{id}/rows.csv?accessType=DOWNLOAD"
DATASETS = {2023: "46xm-urtu", 2024: "sf4k-39ay"}
EXPECTED_ROWS = {2023: 2_125_754, 2024: 2_196_737}
EXPECTED_COLUMNS = 33

raw = {}
for year, dataset_id in DATASETS.items():
    path = RAW_DIR / f"sparcs_{year}.csv"
    if not path.exists():
        print(f"Downloading {year}...")
        RAW_DIR.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(URL.format(id=dataset_id), path)

    raw[year] = pd.read_csv(path, dtype=str, engine="pyarrow")
    assert raw[year].shape == (EXPECTED_ROWS[year], EXPECTED_COLUMNS), (
        f"{path} has shape {raw[year].shape}. If the download was interrupted, "
        "delete the file and run this again."
    )
    print(f"{year}: {raw[year].shape[0]:,} rows, {raw[year].shape[1]} columns")

2023: 2,125,754 rows, 33 columns


2024: 2,196,737 rows, 33 columns


## Step 2: Build the cohort and sample

This step only lines up the columns, creates the target and chooses the rows.
Before each decision, we look at the data.

### Pool the two years

First, do the two files use the same column names?

In [3]:
def snake(name):
    return re.sub(r"[^0-9a-z]+", "_", name.lower()).strip("_")


columns = {year: {snake(c) for c in df.columns} for year, df in raw.items()}
print("Only in 2023:", sorted(columns[2023] - columns[2024]))
print("Only in 2024:", sorted(columns[2024] - columns[2023]))
print("Year column present:", "discharge_year" in columns[2023] & columns[2024])

Only in 2023: ['hospital_service_area', 'zip_code_3_digits']
Only in 2024: ['health_service_area', 'zip_code']
Year column present: True


Two columns have different names, so we rename them. Without this, each would
split into two half-empty columns. The files already have a year column, so we
don't need to add one.

In [4]:
RENAMES = {
    "hospital_service_area": "health_service_area",
    "zip_code_3_digits": "zip_code",
}

frames = [df.rename(columns=snake).rename(columns=RENAMES) for df in raw.values()]
assert set(frames[0].columns) == set(frames[1].columns)

cohort = pd.concat(frames, ignore_index=True)
del raw, frames
print(f"{len(cohort):,} stays")

4,322,491 stays


### Create the target

Length of stay is stored as text. Which values aren't plain whole numbers?

In [5]:
is_number = cohort["length_of_stay"].str.fullmatch(r"\d+")
print(cohort.loc[~is_number, "length_of_stay"].value_counts().to_string())

length_of_stay
120+     2404
120 +    2290


The only exception is the top value, spelled "120 +" in 2023 and "120+" in
2024. We treat both as 120. A long stay is more than 7 days, so $y = 1$ if
length of stay is at least 8.

In [6]:
los_text = cohort["length_of_stay"].str.replace(" ", "").replace("120+", "120")
cohort["los"] = pd.to_numeric(los_text)
cohort["y"] = (cohort["los"] >= 8).astype(int)

print(f"Length of stay ranges from {cohort['los'].min()} to {cohort['los'].max()}")
print(f"Long stays: {cohort['y'].mean():.1%} of all stays")

Length of stay ranges from 1 to 120
Long stays: 20.0% of all stays


In [7]:
def long_stay_table(df, by):
    """Number of stays and long-stay rate for each level of a column."""
    table = df.groupby(by, dropna=False)["y"].agg(stays="size", long_stay_rate="mean")
    return table.sort_values("stays", ascending=False).round(3)

### Admission type

Which kinds of admission are in the data?

In [8]:
print(long_stay_table(cohort, "type_of_admission").to_string())

                     stays  long_stay_rate
type_of_admission                         
Emergency          2870379           0.229
Elective            712698           0.143
Newborn             392468           0.054
Urgent              322676           0.242
Trauma               18117           0.274
Not Available         6153           0.349


We're predicting long stays for unplanned admissions, so we keep emergency,
urgent and trauma. Elective stays are largely planned, newborn stays work
differently, and "Not Available" doesn't tell us the type.

In [9]:
KEEP_ADMISSION_TYPES = ["Emergency", "Urgent", "Trauma"]

cohort = cohort[cohort["type_of_admission"].isin(KEEP_ADMISSION_TYPES)]
print(f"{len(cohort):,} stays remain")

3,211,172 stays remain


### Newborns

The admission type filter removes stays labeled "Newborn". Are any newborns
left under other admission types? APR MDC code 15 is the newborn diagnostic
group.

In [10]:
newborns = cohort[cohort["apr_mdc_code"] == "15"]
print(f"{len(newborns):,} stays, {newborns['y'].mean():.1%} long")
print()
print(long_stay_table(newborns, "type_of_admission").to_string())
print()
print("Top diagnoses:")
diagnoses = newborns["ccsr_diagnosis_description"].str.capitalize()
print(diagnoses.value_counts().head(5).to_string())

10,250 stays, 19.7% long

                   stays  long_stay_rate
type_of_admission                       
Emergency           6015           0.080
Urgent              4217           0.364
Trauma                18           0.167

Top diagnoses:
ccsr_diagnosis_description
Hemolytic jaundice and perinatal jaundice               3389
Liveborn                                                2730
Other specified and unspecified perinatal conditions     895
Respiratory distress syndrome                            589
Respiratory perinatal condition                          463


Some are, though they're a small group. Our plan leaves newborns out because
their stays work differently, so we drop this group too.

In [11]:
cohort = cohort[cohort["apr_mdc_code"] != "15"]
print(f"{len(cohort):,} stays remain")

3,200,922 stays remain


### Redacted records

There's no column flagging redacted records, but `facility_name` says
"Redacted" on some rows. What's blank on those rows?

In [12]:
LOCATION_COLUMNS = [
    "permanent_facility_id",
    "hospital_county",
    "health_service_area",
    "zip_code",
]

is_redacted = cohort["facility_name"].str.contains("Redacted", na=False)
print(
    f"{is_redacted.sum():,} redacted stays, {cohort.loc[is_redacted, 'y'].mean():.1%} long"
)
print(f"Blank facility ID: {cohort['permanent_facility_id'].isna().sum():,} stays")
print()
print("Share of rows blank:")
blank = cohort[LOCATION_COLUMNS].isna().groupby(is_redacted).mean().round(3)
print(blank.rename_axis("redacted").to_string())
print()
print("Top diagnostic groups in redacted stays:")
print(cohort.loc[is_redacted, "apr_mdc_code"].value_counts().head(3).to_string())

8,606 redacted stays, 7.8% long
Blank facility ID: 8,606 stays

Share of rows blank:


          permanent_facility_id  hospital_county  health_service_area  zip_code
redacted                                                                       
False                       0.0              0.0                  0.0     0.016
True                        1.0              1.0                  1.0     1.000

Top diagnostic groups in redacted stays:
apr_mdc_code
14    5872
19     396
04     318


The publisher blanks the location on these rows for privacy, and its
documentation describes this as extra de-identification for abortion records.
Every redacted row has no facility, county, service area or ZIP. A real hospital
always knows which facility it is, so these rows don't fit how the model would
be used, and we drop them.

This removes mostly pregnancy-related stays (MDC 14), so the paper needs to say
so.

In [13]:
cohort = cohort[cohort["permanent_facility_id"].notna()]
print(f"{len(cohort):,} stays remain")

3,192,316 stays remain


### Specialty facilities

We want a model for typical hospitals. The publisher lists 10 facilities whose
discharges are mostly rehab or life support. How many of their stays are left?

In [14]:
SPECIALTY_FACILITIES = [
    "000340",
    "000775",
    "000831",
    "000968",
    "001046",
    "001138",
    "001460",
    "001486",
    "009431",
    "010223",
]

is_specialty = cohort["permanent_facility_id"].isin(SPECIALTY_FACILITIES)
print(f"{is_specialty.sum():,} stays, {cohort.loc[is_specialty, 'y'].mean():.1%} long")
print()
print(long_stay_table(cohort[is_specialty], "permanent_facility_id").to_string())

176 stays, 59.7% long

                       stays  long_stay_rate
permanent_facility_id                       
001046                   106           0.915
000968                    63           0.063
000831                     4           0.750
001460                     2           0.000
000340                     1           1.000


Very few, and only five of the 10 facilities have any stays left. We drop them
so that the rule is simple: the publisher's list defines a specialty facility.

In [15]:
cohort = cohort[~is_specialty]
print(f"{len(cohort):,} stays remain")

3,192,140 stays remain


### Kept on purpose

These groups might look like candidates for removal, but we keep all of them.

In [16]:
died = cohort["patient_disposition"] == "Expired"
print(
    f"Patients who died: {died.sum():,} stays, {cohort.loc[died, 'y'].mean():.1%} long"
)
print(f"Exact duplicate rows: {cohort.duplicated().sum():,} extra rows")

is_blank_zip = cohort["zip_code"].isna()
print(f"Blank ZIP code: {is_blank_zip.sum():,} stays")
print()

is_behavioral = cohort["apr_mdc_code"].isin(["19", "20"])
facilities = cohort.assign(behavioral=is_behavioral).groupby("permanent_facility_id")
facilities = facilities.agg(
    stays=("y", "size"),
    long_stay_rate=("y", "mean"),
    behavioral_share=("behavioral", "mean"),
    name=("facility_name", "first"),
)
print("Highest long-stay rates among facilities with at least 200 stays:")
highest = facilities[facilities["stays"] >= 200]
highest = highest.sort_values("long_stay_rate", ascending=False).head(6).round(3)
print(highest.to_string())

Patients who died: 98,675 stays, 45.4% long


Exact duplicate rows: 184 extra rows
Blank ZIP code: 50,889 stays

Highest long-stay rates among facilities with at least 200 stays:
                       stays  long_stay_rate  behavioral_share                                                        name
permanent_facility_id                                                                                                     
000482                   434           0.836             0.355           St. Mary's Healthcare - Amsterdam Memorial Campus
001047                  4319           0.830             0.987  New York-Presbyterian Westchester Behavioral Health Center
001133                  4405           0.770             0.999          St. Joseph's MC-St. Vincent's Westchester Division
015437                  1215           0.725             0.991                     MOUNT SINAI -  BEHAVIORAL HEALTH CENTER
009059                   871           0.719             0.983                     Crouse Hospital - Commonwealth Division
000412

- **Patients who died:** death isn't known at admission, so removing them would
  use future information. Their stays are cut short, which the paper should say.
- **Exact duplicate rows:** with no patient ID, we can't tell whether these are
  errors or separate stays that look alike.
- **Blank ZIP codes:** ZIP is the patient's home area, not the hospital's
  location, so we handle these blanks during cleaning.
- **Facilities with many long stays:** these aren't on the publisher's list.
  `behavioral_share` is the share of stays for mental health or substance use
  (MDC 19 and 20). Most of these sites are nearly all behavioral, but not all of
  them. Choosing sites by long-stay rate would select on the outcome we're
  predicting, so they stay in and the paper notes it.

### Check the cohort

Stop with an error if the cohort isn't the expected size.

In [17]:
EXPECTED_COHORT = {"2023": 1_562_648, "2024": 1_629_492}

stays_by_year = cohort["discharge_year"].value_counts().to_dict()
assert stays_by_year == EXPECTED_COHORT, stays_by_year
assert len(cohort) == 3_192_140, len(cohort)
assert cohort["los"].notna().all()
assert round(cohort["y"].mean(), 4) == 0.2309, cohort["y"].mean()

print(f"Cohort: {len(cohort):,} stays, {cohort['y'].mean():.2%} long")
print(pd.Series(stays_by_year).to_string())

Cohort: 3,192,140 stays, 23.09% long
2024    1629492
2023    1562648


### Draw the sample

To keep the slower models practical to train, all models use one simple random
sample of 500,000 stays from the roughly 3.2 million in the cohort.

In [18]:
SAMPLE_SIZE = 500_000

sample = cohort.sample(n=SAMPLE_SIZE, random_state=SEED)
print(f"Sample: {len(sample):,} stays, {sample['y'].mean():.2%} long")

Sample: 500,000 stays, 23.06% long


Does the sample look like the cohort? The two columns below are the percentage
of stays at each level.

In [19]:
for column in ["discharge_year", "type_of_admission", "payment_typology_1", "race"]:
    shares = pd.DataFrame(
        {
            "cohort": cohort[column].value_counts(normalize=True, dropna=False),
            "sample": sample[column].value_counts(normalize=True, dropna=False),
        }
    )
    print((shares * 100).round(2).to_string())
    print()

                cohort  sample
discharge_year                
2024             51.05   51.08
2023             48.95   48.92

                   cohort  sample
type_of_admission                
Emergency           89.49   89.45
Urgent               9.94    9.97
Trauma               0.57    0.57

                           cohort  sample
payment_typology_1                       
Blue Cross/Blue Shield       7.56    7.57
Department of Corrections    0.09    0.08
Federal/State/Local/VA       0.99    1.02
Managed Care, Unspecified    1.21    1.19
Medicaid                    27.73   27.63
Medicare                    48.63   48.74
Miscellaneous/Other          1.00    0.99
Private Health Insurance    11.45   11.48
Self-Pay                     1.34    1.30
NaN                          0.00    0.00

                        cohort  sample
race                                  
White                    52.18   52.30
Other Race               26.68   26.53
Black/African American   19.83   19.87
Mult